# The weekly numbers — one table

The head of sales runs a trading meeting every Monday. Before those numbers move to a dashboard, she wants each one
answered once, by hand, by someone who can say what it counts. Nine questions, one table:
`data/raw/online_retail.parquet`, every invoice line of the shop, one row per line of an invoice, `Price` in pounds
per unit.

Finance's brief defines every word the questions use. It is the definition; the data cannot tell it to you.

## The brief

> **The period** is the whole file: 1 December 2009 to 9 December 2010.
>
> **Sales invoices.** A sales invoice has a six-digit number. An invoice number that starts with a letter is not a
> sale: `C` marks a cancellation, and `A` an accounting adjustment.
>
> **Revenue** is the sum of `Quantity * Price` over the lines of sales invoices.
>
> **Units sold** of a product is the sum of `Quantity` over its lines on sales invoices that carry a price above zero.
>
> **A product** is an item the shop stocks and sells, identified by its `StockCode`. Postage, carriage, discounts,
> manual adjustments, fees, bank charges, samples, and test lines have stock codes too. They are not products.
>
> **Identified revenue** is revenue from lines that carry a `Customer ID`. Lines with no `Customer ID` are
> *unidentified*: part of revenue, and part of no per-customer figure. **Average revenue per identified customer** is
> identified revenue divided by the number of distinct `Customer ID`s on those same lines.
>
> **Manual adjustments** are the lines whose `Description` is `Manual`.
>
> **Country** is the `Country` column as the file writes it. *Outside the United Kingdom* is every line whose
> `Country` is not `United Kingdom`.
>
> **A week** starts on a Monday: `date_trunc('week', InvoiceDate)` gives that Monday.
>
> **The lines as they are.** Every figure is computed on the lines the file has. Do not delete, merge, or de-duplicate
> lines unless a sentence above says so.


## How to answer each question

Four parts, every time, in the cells under the question:

1. **The sentence.** The number as a sentence: its filter, its measure, its population. Write it in the markdown cell.
2. **The query.** First a comment with the number of rows you expect. Then the query. Where the brief fixes the
   count (one row per month, a top five), write it; where only the file can (question 4's countries), write a
   range and the reason, and after the query one word on whether the file agreed.
3. **The number.** Copy it into the markdown cell, with its unit.
4. **The check.** One query that would fail if the number were wrong: an identity that must hold, a count that must be
   zero, or the same number reached another way. Say in one line what it shows. Compare sums of money **to the
   penny** — `abs(a - b) < 0.005`, or both rounded to the cent — never with `==`: two correct sums of the same lines
   can differ in the tenth decimal place. A ranked list is not a sum: count its rows, reach its top row a second way,
   and show the **boundary**, one thing the rule kept beside one it excluded, so the reader sees the rule at its edge
   (for question 5: the units of one product in the ten, and the units of one code the product rule threw out).

**What Python may do with a number:** add, subtract, divide and compare the numbers you fetched with
`.fetchone()[0]`. A sum, a count or an average *over rows* is SQL — a query, or a view — never `.sum()` or `.mean()`
on a dataframe. pandas displays; it does not compute the answer.

The worked example below shows all four parts on a question that is not one of the nine.

**Questions 1 to 5, 7 and 8 are the homework.** Questions 6 and 9 are stretch (README): do them after the seven are
answered and checked, or not at all. They are not graded.

**Every code cell starts with its ID**: `I1`, `I2`, … for the inspection queries, `S` for the sales view, `Q1` … `Q9`
for each question's query, `Q1 check` … `Q9 check` for its check, `R` for the reconciliation. Your trap log in
`DIAGNOSIS.md` cites cells by ID instead of pasting them again.

When a check fails, or a number looks wrong, **stop**: paste the query and what it returned into your trap log before
you change anything. Once you fix the query, the notebook no longer shows the wrong number. The README says why.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between: count it, then census it by group

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

---

# Your work starts here

## 0. Inspect first

Block 1's four queries on `data/raw/online_retail.parquet`, before any question: `DESCRIBE`, `SUMMARIZE`, the key
test, and the census of every column a question depends on. What to look at, one cell each, in this order:

- the columns and their types; what is missing, and how much, per column;
- the key test: rows against distinct (`Invoice`, `StockCode`);
- the first character of `Invoice`, with `SUM(Quantity * Price)` beside each count;
- the sign of `Price` and the sign of `Quantity`: how many lines are negative, zero, positive;
- the stock codes: real product codes look alike, so count the codes that do not, and read them;
- `Description`: how many are missing, how many say `Manual`;
- `Customer ID`: how many lines have none;
- `Country`: the census, read to the end.

A census of a few dozen values you read to the end. A column with thousands of distinct values (a code, a
description) you do not read by eye: pandas prints the first and last rows of a result longer than 400 with `...`
between, and the middle is where the surprises hide. Count it (`COUNT(DISTINCT …)`), census it in the shape the
question uses, and then look at any group that surprises you: `SELECT *`, its `WHERE`, `LIMIT 20`. **Write the first
query in the cell below, and add one cell per query after it**, each starting with its ID — `# I1`, `# I2`, … — and
note in the trap log what surprised you, citing the cell. These cells never change, so their output stays in the
notebook as evidence.

In [ ]:
# I1. Write your first inspection query in this cell. One query per cell; each new cell starts with its ID: I2, I3, …

### The lines that count

The brief's *sales invoices*, written once as a view (README, *The order of work*, step 2). Questions 2 to 9 read
from it. Question 1 is about the table as the file holds it, before any brief: it reads
`data/raw/online_retail.parquet` directly.

In [ ]:
# S. Write the view in this cell: CREATE OR REPLACE VIEW sales AS ...

## A worked example, not one of the nine: how many sales invoices does the file hold?

Every question below is answered in this shape. The two cells run as they are; read them before you write Q1.

**Sentence.** The number of distinct invoice numbers on the lines of sales invoices (six-digit numbers: not `C`, not
`A`), over the whole file. Filter: sales invoices. Measure: distinct `Invoice`. Population: every line of the file.

**Number.** **24,221** sales invoices.

**Check.** Every invoice has one first character, so the census of the first character splits all invoices into
groups that add back up: the `4` and `5` groups together are the 24,221 (5,261 + 18,960), and all four groups are
the file's 28,816 distinct invoices. A count is a whole number, so `==` is right here.

In [ ]:
# W. Expect one number.
con.sql("""
    SELECT COUNT(DISTINCT Invoice) AS sales_invoices
    FROM 'data/raw/online_retail.parquet'
    WHERE left(Invoice, 1) != 'C' AND left(Invoice, 1) != 'A'
""").df()

In [ ]:
# W check: the same count a second way, by the first character; the digit groups must add up to it.
by_first = con.sql("""
    SELECT left(Invoice, 1) AS first_character, COUNT(DISTINCT Invoice) AS invoices
    FROM 'data/raw/online_retail.parquet'
    GROUP BY first_character
    ORDER BY first_character
""").df()
print(by_first)
digits = con.sql("""
    SELECT SUM(invoices) FROM (
        SELECT left(Invoice, 1) AS first_character, COUNT(DISTINCT Invoice) AS invoices
        FROM 'data/raw/online_retail.parquet'
        GROUP BY first_character
    ) WHERE first_character IN ('4', '5')
""").fetchone()[0]
sales = con.sql("""
    SELECT COUNT(DISTINCT Invoice) FROM 'data/raw/online_retail.parquet'
    WHERE left(Invoice, 1) != 'C' AND left(Invoice, 1) != 'A'
""").fetchone()[0]
print("4 + 5 groups:", digits, "  sales invoices:", sales, "  equal:", digits == sales)

## 1. What is one row of this table, and what identifies it?

State the grain in one sentence. Then prove whether (`Invoice`, `StockCode`) is a key. If it is not: how many rows would disappear if exact duplicates — rows identical in every column — were collapsed to one? (That is, rows minus distinct rows.) This question is about the file, not the brief: read `data/raw/online_retail.parquet`, not `sales`.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q1. How many rows do you expect?
# Your query.

In [ ]:
# Q1 check.

## 2. What was revenue over the whole file?

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q2. How many rows do you expect?
# Your query.

In [ ]:
# Q2 check.

## 3. Revenue by month

One row per month, in date order.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q3. How many rows do you expect?
# Your query.

In [ ]:
# Q3 check.

## 4. Revenue by country outside the United Kingdom

Largest first. How much came from Ireland? Check the census for how the file writes the country you are asked about.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q4. How many rows do you expect?
# Your query.

In [ ]:
# Q4 check.

## 5. The ten products with the most units sold

Stock code, a description, and units, most first.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q5. How many rows do you expect?
# Your query.

In [ ]:
# Q5 check.

## 6. The ten products that brought in the most revenue (stretch)

Stock code, a description, and revenue, most first.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q6. How many rows do you expect?
# Your query.

In [ ]:
# Q6 check.

## 7. Average revenue per identified customer

With the unidentified revenue beside it, and what fraction of question 2's revenue it is: two numbers and one division.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q7. How many rows do you expect?
# Your query.

In [ ]:
# Q7 check.

## 8. Revenue without the manual adjustments, and the number of sales lines it comes from

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q8. How many rows do you expect?
# Your query.

In [ ]:
# Q8 check.

## 9. The five weeks with the highest revenue (stretch)

Each week (its Monday) and its revenue, highest first.

**Sentence.** _(the filter, the measure, the population)_

**Number.** _(with its unit, once the query has run)_

**Check.** _(one line: what it shows)_

In [ ]:
# Q9. How many rows do you expect?
# Your query.

In [ ]:
# Q9 check.

## The reconciliation

Three computations of one number. Revenue by month, added up; revenue by country — **every** country, the United
Kingdom included — added up; and question 2's revenue. Revenue is additive, so all three must agree to the penny:
`abs(a - b) < 0.005`, or both rounded to the cent. **Not `==`**: the three are correct sums of the same lines, added in
different orders, and they can differ in the tenth decimal place. Print the three, and assert that they agree.

In [ ]:
# R. By month, by country, the grand total, and the two differences.

---

Before you submit: *Restart*, then *Run All*. Every number above must come from a cell that ran in order, from the top.